# Product Documents Enrichment Pipeline

This notebook enriches product data by combining product metadata with their corresponding documentation. It creates a structured indexed document that can be used for search, retrieval, or AI applications.

## Workflow Overview
1. Load product metadata from `products` table
2. Load product documentation from `product_docs` table
3. Join both datasets on product name
4. Create structured indexed documents with XML-like tags
5. Save the enriched data to `product_docs_combined` table
6. Enable Change Data Feed for tracking changes

---

## Step 1: Load Products Metadata

Load the products table containing basic product information like ID, name, category, and sub-category.

In [0]:
# Import required PySpark functions
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Load products table from Unity Catalog
# Table contains: product_id, product_name, product_category, product_sub_category
products_df = spark.table("agentic_ai.ecommerce.products")

# Display schema to understand the structure
print("Products Table Schema:")
products_df.printSchema()
print(f"\nTotal products: {products_df.count()}")

# Preview sample data
print("\nSample products:")
display(products_df.limit(5))

## Step 2: Load Product Documentation

Load the product documentation table containing detailed descriptions and specifications for each product.

In [0]:
# Load product documentation table from Unity Catalog
# Table contains: product_name, product_doc (detailed product descriptions)
product_docs_df = spark.table("agentic_ai.ecommerce.product_docs")

# Display schema to understand the structure
print("Product Docs Table Schema:")
product_docs_df.printSchema()
print(f"\nTotal documents: {product_docs_df.count()}")

# Preview sample documents
print("\nSample documents:")
display(product_docs_df.limit(5))

## Step 3: Join Products with Documentation

Perform an inner join to combine product metadata with their documentation, ensuring we only keep products that have both metadata and documentation.

In [0]:
# Join products with their documentation
# Using inner join to keep only products that have both metadata and documentation
# Join key: product_name (common column in both tables)
joined_df = products_df.join(
    product_docs_df,
    on="product_name",
    how="inner"
)

# Verify the join results
print("Joined Table Schema:")
joined_df.printSchema()
print(f"\nTotal joined records: {joined_df.count()}")

# Preview combined data
print("\nSample joined data:")
display(joined_df.limit(5))

## Step 4: Create Indexed Documents

Create a structured indexed document by combining product metadata and documentation with XML-like tags. This format makes it easier for:
- Search and retrieval systems
- LLM/AI applications to parse product information
- Maintaining semantic structure in text embeddings

In [0]:
# Create indexed document with XML-like structure
# This format helps with:
# - Semantic search and retrieval
# - LLM parsing and understanding
# - Structured embeddings for vector databases
indexed_df = joined_df.withColumn(
    "indexed_doc",
    F.concat(
        # Add product category tags
        F.lit("<product_category>"),
        F.col("product_category"),
        F.lit("</product_category>\n"),
        # Add product sub-category tags
        F.lit("<product_sub_category>"),
        F.col("product_sub_category"),
        F.lit("</product_sub_category>\n"),
        # Add product name tags
        F.lit("<product_name>"),
        F.col("product_name"),
        F.lit("</product_name>\n"),
        # Add full product documentation
        F.lit("<product_doc>\n"),
        F.col("product_doc"),
        F.lit("\n</product_doc>")
    )
)

print("Indexed document column created successfully!")

# Preview the indexed document structure
print("\nSample indexed document (first 500 characters):")
sample_indexed = indexed_df.select("product_name", "indexed_doc").first()
print(f"\nProduct: {sample_indexed['product_name']}")
print(f"\nIndexed Doc:\n{sample_indexed['indexed_doc'][:500]}...")

## Step 5: Prepare Final Dataset

Select the relevant columns for the final output table, including both the original fields and the newly created indexed document.

In [0]:
# Select columns for the final output table
# Includes:
# - Original product metadata (id, name, category, sub_category)
# - Original documentation (product_doc)
# - Enriched indexed document (indexed_doc)
final_df = indexed_df.select(
    "product_id",
    "product_name",
    "product_doc",
    "product_category",
    "product_sub_category",
    "indexed_doc"
)

# Verify final structure before writing
print("Final Table Schema:")
final_df.printSchema()
print(f"\nTotal records: {final_df.count()}")

# Preview final dataset
print("\nSample final data:")
display(final_df.limit(5))

## Step 6: Save to Unity Catalog

Write the enriched data to a Delta table in Unity Catalog. Using `overwrite` mode to ensure a clean write.

In [0]:
# Define target table in Unity Catalog
target_table = "agentic_ai.ecommerce.product_docs_combined"

# Write the DataFrame as a Delta table
# - format("delta"): Use Delta Lake format for ACID transactions and time travel
# - mode("overwrite"): Replace the table if it already exists
# - overwriteSchema: Allow schema changes if table exists
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print(f"Table saved successfully: {target_table}")
print(f"\nTotal records written: {spark.table(target_table).count()}")

## Step 7: Verify the Output

Verify that the data was written correctly by reading it back and checking the schema and record count.

In [0]:
# Read back the saved table to verify
verify_df = spark.table("agentic_ai.ecommerce.product_docs_combined")

# Confirm data was written correctly
print("Verification Results:")
print(f"Total records: {verify_df.count()}")
print("\nTable Schema:")
verify_df.printSchema()

# Preview the saved data
print("\nSample Records:")
display(verify_df.limit(5))

## Step 8: Enable Change Data Feed

Enable Change Data Feed (CDF) on the table to track all future changes (inserts, updates, deletes). This is useful for:
- Incremental processing in downstream pipelines
- Audit trails and compliance
- Real-time data synchronization

In [0]:
# Enable Change Data Feed (CDF) on the table
# CDF tracks all changes (inserts, updates, deletes) and stores them separately
# Benefits:
# - Incremental processing: Read only changed data in downstream pipelines
# - Audit trail: Track who changed what and when
# - CDC patterns: Replicate changes to other systems
spark.sql("""
ALTER TABLE agentic_ai.ecommerce.product_docs_combined
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

---

## Summary

✅ **Completed Steps:**
1. Loaded product metadata from `agentic_ai.ecommerce.products`
2. Loaded product documentation from `agentic_ai.ecommerce.product_docs`
3. Joined datasets on `product_name`
4. Created structured indexed documents with XML-like tags
5. Saved enriched data to `agentic_ai.ecommerce.product_docs_combined`
6. Enabled Change Data Feed for tracking future changes

**Output Table:** `agentic_ai.ecommerce.product_docs_combined`

**Key Columns:**
- `product_id` - Unique product identifier
- `product_name` - Product name
- `product_category` - Main category
- `product_sub_category` - Sub-category classification
- `product_doc` - Original product documentation
- `indexed_doc` - **Structured document with XML tags** (ready for search/AI applications)

## Next Steps

This enriched data can now be used for:
- **Vector Search:** Create embeddings from `indexed_doc` for semantic search
- **RAG Applications:** Use structured content for retrieval-augmented generation
- **AI Agents:** Parse XML tags to extract specific product attributes
- **Analytics:** Query and analyze product information with full documentation context